# SmartCash Detector - Environment Setup

This notebook sets up the development environment and validates all required dependencies for the SmartCash Detector project.

## Overview
1. Environment validation
2. Package installation
3. GPU verification
4. Project path setup
5. Import validation

In [1]:
import sys
import os
from pathlib import Path
import subprocess
from tqdm.notebook import tqdm
import time

# Add project root to path
project_root = Path().absolute().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## 1. Environment Validation

In [2]:
def check_python_version():
    """Verify Python version >= 3.8"""
    version = sys.version_info
    if version.major < 3 or (version.major == 3 and version.minor < 8):
        raise RuntimeError(f"Python 3.8+ required, but found {version.major}.{version.minor}")
    return f"{version.major}.{version.minor}.{version.micro}"

print(f"✅ Python version: {check_python_version()}")

✅ Python version: 3.12.8


## 2. Package Installation

In [3]:
requirements = [
    'torch>=2.0.0',
    'torchvision>=0.15.0',
    'opencv-python>=4.8.0',
    'albumentations>=1.3.1',
    'efficientnet_pytorch>=0.7.1',
    'shapely>=2.0.0',  # For polygon operations
    'tqdm>=4.65.0',
    'pyyaml>=6.0.1',
    'termcolor>=2.3.0'
]

def install_package(package):
    """Install a single package using pip"""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("📦 Installing required packages...")
for package in tqdm(requirements, desc="Installing packages"):
    try:
        install_package(package)
    except Exception as e:
        print(f"❌ Failed to install {package}: {str(e)}")

📦 Installing required packages...


Installing packages:   0%|          | 0/9 [00:00<?, ?it/s]

## 3. GPU Verification

In [4]:
import torch

def check_gpu():
    """Verify GPU availability and CUDA support"""
    if torch.cuda.is_available():
        device_count = torch.cuda.device_count()
        devices = [torch.cuda.get_device_name(i) for i in range(device_count)]
        cuda_version = torch.version.cuda
        return {
            'available': True,
            'device_count': device_count,
            'devices': devices,
            'cuda_version': cuda_version
        }
    return {'available': False}

gpu_info = check_gpu()
if gpu_info['available']:
    print(f"✅ GPU detected:")
    print(f"   • CUDA version: {gpu_info['cuda_version']}")
    print(f"   • Devices ({gpu_info['device_count']}):{' '.join(gpu_info['devices'])}")
else:
    print("⚠️ No GPU detected - using CPU only")

⚠️ No GPU detected - using CPU only


## 4. Project Path Setup

In [5]:
def setup_project_dirs():
    """Create required project directories"""
    dirs = [
        project_root / 'data' / 'rupiah' / split / subdir
        for split in ['train', 'val', 'test']
        for subdir in ['images', 'labels']
    ]
    dirs.extend([
        project_root / d 
        for d in ['weights', 'exports', 'logs']
    ])
    
    for d in tqdm(dirs, desc="Creating directories"):
        d.mkdir(parents=True, exist_ok=True)
        
setup_project_dirs()
print("✅ Project directories created")

Creating directories:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Project directories created


## 5. Import Validation

In [6]:
def validate_imports():
    """Validate all required imports"""
    required_imports = [
        ('numpy', 'np'),
        ('torch', None),
        ('torchvision', None),
        ('cv2', None),
        ('albumentations', 'A'),
        ('efficientnet_pytorch', None),
        ('shapely.geometry', None),
        ('yaml', None),
        ('tqdm', None)
    ]
    
    results = []
    for module, alias in tqdm(required_imports, desc="Validating imports"):
        try:
            if alias:
                exec(f"import {module} as {alias}")
            else:
                exec(f"import {module}")
            results.append((module, True, None))
        except Exception as e:
            results.append((module, False, str(e)))
    
    return results

import_results = validate_imports()
print("\n📦 Import Validation Results:")
for module, success, error in import_results:
    status = "✅" if success else "❌"
    print(f"{status} {module}")
    if error:
        print(f"   Error: {error}")

Validating imports:   0%|          | 0/9 [00:00<?, ?it/s]


📦 Import Validation Results:
✅ numpy
✅ torch
✅ torchvision
✅ cv2
✅ albumentations
✅ efficientnet_pytorch
✅ shapely.geometry
✅ yaml
✅ tqdm
